# GridNet-HD — LitePT-S LiDAR-only training on Colab (Phase 1, resume-safe)

เป้าหมาย: เทรน baseline LiDAR-only ด้วย cache ของเราเอง (อ้างอิง: entry `Lite-S-test` บน leaderboard = 72.88 mIoU; entry นี้ไม่มี publication)
Runtime: **A100 80 GB** (เร็วสุด ~15 CU/ชม.) หรือ **L4 24 GB** (ถูกกว่า ช้ากว่า ~2.5 เท่า)

**VRAM**: LitePT-S ใช้ ~18 KB/จุด (รวม backward) → `BATCH × MAX_POINTS` ต้องไม่เกิน ~3.5M จุดบน A100 และ **~0.8M จุดบน L4**
| GPU | MAX_POINTS | EVAL_MAX_POINTS |
|---|---|---|
| A100 80 GB | 600000 (default) | 900000 (default) |
| L4 24 GB | **300000** | **500000** |

**LR**: ตาม recipe ของ LitePT (nuScenes) — blocks ใช้ 0.1× ของ base LR; base 1e-3 สำหรับ batch 2×600k
(รัน v1 ที่ 2e-3 ทุก parameter ลู่ออกเป็น NaN ที่ it ~2350) ถ้าเห็น `skipped N` ใน log บ่อย → ลด LR อีกครึ่ง

**Resume**: กด Run all ซ้ำได้ทุกครั้งที่ session หลุด — เทรนต่อจาก `last.pth` บน Drive อัตโนมัติ
ตั้ง `MAX_HOURS` ให้น้อยกว่าอายุ session เพื่อให้หยุดพร้อม checkpoint เอง · **เปลี่ยน LR/batch = เปลี่ยน RUN_NAME** (resume ข้าม config ไม่ได้)

**ก่อนรัน**: cache ครบ (notebook build_cache) และ `GridNet/code/gridnet_code.zip` บน Drive

In [ ]:
#@title 1. GPU
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
assert torch.cuda.get_device_capability()[0] >= 8, "need L4 / A100"


In [ ]:
#@title 2. Mount Drive + parameters
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/GridNet'
CODE_ZIP = f'{DRIVE}/code/gridnet_code.zip'      #@param {type:"string"}
CACHE_DRIVE = f'{DRIVE}/cache'
RUN_NAME = 'litept_s_lidar_v2'                    #@param {type:"string"}
CKPT_DIR = f'{DRIVE}/ckpt/{RUN_NAME}'
ITERS = 20000        #@param {type:"integer"}
MAX_HOURS = 6.5      #@param {type:"number"}
BATCH = 2            #@param {type:"integer"}
MAX_POINTS = 600000  #@param {type:"integer"}
EVAL_MAX_POINTS = 900000  #@param {type:"integer"}
RADIUS = 25.0        #@param {type:"number"}
LR = 0.001           #@param {type:"number"}
BLOCK_LR_MULT = 0.1  #@param {type:"number"}
EVAL_EVERY = 4000    #@param {type:"integer"}
ZONES_LIMIT = 0      #@param {type:"integer"}
os.makedirs(CKPT_DIR, exist_ok=True)
import torch
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if BATCH * MAX_POINTS * 18e3 > 0.85 * vram_gb * 1e9:
    print(f"WARNING: {BATCH} x {MAX_POINTS:,} points needs ~{BATCH * MAX_POINTS * 18e-6:.0f} GB but this GPU has {vram_gb:.0f} GB -> lower MAX_POINTS")
print("checkpoints ->", CKPT_DIR)


In [ ]:
#@title 3. Environment (restore from Drive tarball if present, else build ~3 min; then save tarball)
import subprocess, os, time
ENV = '/content/env'; TAR = f'{DRIVE}/env/env_py310_torch241_cu124.tar.gz'
t0 = time.time()
if not os.path.exists(f'{ENV}/bin/python'):
    if os.path.exists(TAR):
        subprocess.run(['tar', '-C', '/content', '-xzf', TAR], check=True); print("env restored from Drive")
    else:
        open('/content/colab_env.sh', 'w').write(r'''#!/usr/bin/env bash
# Build a self-contained Python 3.10 + torch 2.4.1/cu124 environment with micromamba.
# Works on Colab (whatever system Python/torch it ships), RunPod and any Linux box with an
# NVIDIA driver >= 12.4. All heavy deps come as prebuilt wheels (no compilation).
#
#   bash colab_env.sh /content/env          # ~8-12 min on Colab
#   /content/env/bin/python -c "import spconv.pytorch, flash_attn, torch_scatter; print('ok')"
set -euo pipefail
ENV=${1:-/content/env}
TORCH=2.4.1
TV=0.19.1
CU=cu124

if [ ! -x "$ENV/bin/python" ]; then
  if ! command -v micromamba >/dev/null 2>&1; then
    mkdir -p /opt/mm && cd /opt/mm
    curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
    export PATH=/opt/mm/bin:$PATH
  fi
  micromamba create -y -q -p "$ENV" -c conda-forge python=3.10 pip git ninja
fi
PIP="$ENV/bin/python -m pip"
$PIP install -q --upgrade pip wheel setuptools packaging psutil
$PIP install -q torch==$TORCH torchvision==$TV --index-url https://download.pytorch.org/whl/$CU
$PIP install -q torch-scatter -f https://data.pyg.org/whl/torch-2.4.0+$CU.html
$PIP install -q spconv-$CU
# flash-attn: setup.py downloads the matching prebuilt wheel (torch 2.4 / cu12 / cp310 exists)
MAX_JOBS=4 $PIP install -q "flash-attn==2.7.4.post1" --no-build-isolation
$PIP install -q "numpy<2.3" scipy addict timm einops plyfile termcolor colorhash tensorboard
"$ENV/bin/python" - << 'EOF'
import torch, spconv.pytorch, torch_scatter, flash_attn
print("env ok | torch", torch.__version__, "| cuda", torch.version.cuda, "| spconv", spconv.__version__,
      "| flash_attn", flash_attn.__version__, "| gpu", torch.cuda.is_available())
EOF
''')
        p = subprocess.Popen(['bash', '/content/colab_env.sh', ENV], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            if 'FutureWarning' not in line and 'def backward' not in line: print(line, end='')
        p.wait(); assert p.returncode == 0
        os.makedirs(f'{DRIVE}/env', exist_ok=True)
        subprocess.run(['tar', '-C', '/content', '-czf', TAR, 'env'], check=True); print("env saved to Drive")
PY = f'{ENV}/bin/python'
print(f"env ready in {(time.time() - t0) / 60:.1f} min")


In [ ]:
#@title 4. Code + LitePT
import subprocess, zipfile, os
os.makedirs('/content/code', exist_ok=True); zipfile.ZipFile(CODE_ZIP).extractall('/content/code')
if not os.path.exists('/content/LitePT'):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/prs-eth/LitePT.git", "/content/LitePT"], check=True)
r = subprocess.run([PY, "-c", "import sys; sys.path.insert(0,'/content/LitePT'); sys.path.insert(0,'/content/code'); "
                    "from litept.model import LitePT; import gridnet.train.engine; print('imports ok')"], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr[-1500:])


In [ ]:
#@title 5. Copy caches Drive → local disk (skips existing; ~1.8 GB per zone). VAL_ONLY = only the 6 val zones (enough for cells 7-8; ~11 GB instead of ~46 GB)
VAL_ONLY = False   #@param {type:"boolean"}
import os, shutil, time
TRAIN = ["t1z6a","t1z6b","t2z5","t3z3","t3z6","t3z7","t5a1","t5a3","t5a4","t5a5","t5b2","t5b3","t5b4","t5b6","t5c1","t5c2","t5c3","t6z2","t6z3","t6z4","t6z6"]
VAL = ["t1z5b","t1z8","t3z4","t4z1","t5b1","t5b5"]
want = [z for z in TRAIN + VAL if os.path.exists(f'{CACHE_DRIVE}/{z}/points.npz')]
if VAL_ONLY: want = [z for z in want if z in VAL]
if ZONES_LIMIT: want = [z for z in want if z in VAL][:2] + [z for z in want if z in TRAIN][:ZONES_LIMIT]
missing = [z for z in TRAIN + VAL if z not in want]
print(f"using {len(want)} zones; missing from Drive: {missing}")
t0 = time.time()
for z in want:
    dst = f'/content/cache/{z}/points.npz'
    if not os.path.exists(dst):
        os.makedirs(f'/content/cache/{z}', exist_ok=True); shutil.copy(f'{CACHE_DRIVE}/{z}/points.npz', dst)
print(f"caches ready in {(time.time() - t0) / 60:.1f} min")


In [ ]:
#@title 6. Train (resumes automatically from last.pth)
import subprocess, time, os
cmd = [PY, '/content/code/scripts/train_litept.py', '--cache', '/content/cache', '--ckpt-dir', CKPT_DIR, '--litept', '/content/LitePT',
       '--iters', str(ITERS), '--max-hours', str(MAX_HOURS), '--batch', str(BATCH), '--max-points', str(MAX_POINTS),
       '--eval-max-points', str(EVAL_MAX_POINTS), '--radius', str(RADIUS), '--eval-every', str(EVAL_EVERY),
       '--lr', str(LR), '--block-lr-mult', str(BLOCK_LR_MULT)]
if ZONES_LIMIT:
    present = sorted(os.listdir('/content/cache'))
    cmd += ['--train-zones', ','.join(z for z in present if z not in VAL), '--val-zones', ','.join(z for z in present if z in VAL)]
env = dict(os.environ, PYTHONPATH='/content/code', PYTHONWARNINGS='ignore', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
t0 = time.time()
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
try:
    for line in p.stdout:
        if 'FutureWarning' not in line and 'def backward' not in line and '_TORCH_CUSTOM_FWD' not in line: print(line, end='')
    p.wait(); print(f"exit {p.returncode} after {(time.time() - t0) / 3600:.2f} h")
except KeyboardInterrupt:
    # the stop button only interrupts this cell: kill the trainer too, or a second run would share the GPU with it
    p.terminate(); p.wait()
    print(f"stopped by user after {(time.time() - t0) / 60:.1f} min; last.pth on Drive is kept -> re-run this cell to resume")


In [ ]:
#@title 7. Evaluate best checkpoint on the val zones
import subprocess, json, os
env = dict(os.environ, PYTHONPATH='/content/code', PYTHONWARNINGS='ignore', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
cmd = [PY, '/content/code/scripts/train_litept.py', '--cache', '/content/cache', '--ckpt-dir', CKPT_DIR, '--litept', '/content/LitePT',
       '--radius', str(RADIUS), '--eval-max-points', str(EVAL_MAX_POINTS), '--eval-only']
if ZONES_LIMIT:
    present = sorted(os.listdir('/content/cache'))
    cmd += ['--train-zones', ','.join(z for z in present if z not in VAL), '--val-zones', ','.join(z for z in present if z in VAL)]
r = subprocess.run(cmd, capture_output=True, text=True, env=env)
print('\n'.join(l for l in r.stdout.splitlines() if 'Warning' not in l)[-3000:], r.stderr[-800:])
if os.path.exists(f'{CKPT_DIR}/eval.json'):
    print(json.dumps(json.load(open(f'{CKPT_DIR}/eval.json')), indent=1))
else:
    print("no eval.json -> training did not produce a checkpoint; see cell 6")


In [ ]:
#@title 8. Figures from best.pth: mIoU per zone + confusion matrix + qualitative renders (~20-30 min on A100; needs cells 1-5)
import subprocess, os, sys, time
QUAL_ZONES = "t5b5,t4z1"   #@param {type:"string"}
PRED = f'{CKPT_DIR}/pred'; FIG = f'{CKPT_DIR}/figures'; os.makedirs(FIG, exist_ok=True)
env = dict(os.environ, PYTHONPATH='/content/code', PYTHONWARNINGS='ignore', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
def run(cmd):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout:
        if 'Warning' not in line and 'def backward' not in line and '_TORCH_CUSTOM_FWD' not in line: print(line, end='')
    p.wait(); return p.returncode
t0 = time.time()
cmd = [PY, '/content/code/scripts/train_litept.py', '--cache', '/content/cache', '--ckpt-dir', CKPT_DIR, '--litept', '/content/LitePT',
       '--radius', str(RADIUS), '--eval-max-points', str(EVAL_MAX_POINTS), '--eval-only', '--save-pred', PRED]
if ZONES_LIMIT:
    present = sorted(os.listdir('/content/cache'))
    cmd += ['--train-zones', ','.join(z for z in present if z not in VAL), '--val-zones', ','.join(z for z in present if z in VAL)]
assert run(cmd) == 0, "eval failed (see output above)"
# the figure scripts need only numpy + matplotlib -> run them with Colab's own python
run([sys.executable, '/content/code/scripts/plot_eval.py', '--eval', f'{CKPT_DIR}/eval.json', '--out', f'{FIG}/eval_val.png'])
for z in [s.strip() for s in QUAL_ZONES.split(',') if s.strip()]:
    if os.path.exists(f'{PRED}/{z}_pred.npy'):
        run([sys.executable, '/content/code/scripts/render_predictions.py', '--cache', '/content/cache', '--zone', z,
             '--pred', f'{PRED}/{z}_pred.npy', '--out', f'{FIG}/qual_{z}'])
    else:
        print(f"no prediction for {z} (not a val zone or not cached)")
print(f"done in {(time.time() - t0) / 60:.1f} min -> figures on Drive: {FIG}")


## หมายเหตุ
- `ZONES_LIMIT > 0` = โหมดทดสอบ: ใช้ train zones แค่ N zone + val 2 zone (เร็ว ใช้ตรวจ pipeline ก่อนรันเต็ม)
- loss/val mIoU ทุกครั้งถูกเขียนที่ `<CKPT_DIR>/log.csv`; config ที่ `config.json`
- จบแล้ว **Disconnect and delete runtime**